In [1]:

# Метод 1: TF‑IDF + LogisticRegression
!pip install -q scikit-learn pandas

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:


import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/LLMTrace_4class_20k.csv')
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0,3] else 1)
df['is_clean'] = df['label'].isin([0,1])

df['stratify'] = df['binary_label'].astype(str) + '_' + df['style']
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df['stratify'])
train_full = df.loc[train_idx].copy()
test_full = df.loc[test_idx].copy()

class TfidfLogReg:
    def __init__(self):
        self.vec = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
        self.clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
    def fit(self, X, y):
        X_vec = self.vec.fit_transform(X)
        self.clf.fit(X_vec, y)
    def predict(self, X):
        return self.clf.predict(self.vec.transform(X))

def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy': accuracy_score(y, pred),
        'f1_AI': f1_score(y, pred, pos_label=1),
        'f1_Human': f1_score(y, pred, pos_label=0)
    }

genres = ['literature', 'scientific', 'publicistic']
results = []

train_clean = train_full[train_full['is_clean']]
test_clean = test_full[test_full['is_clean']]
for genre in genres:
    train_genre = train_clean[train_clean['style']==genre]
    if len(train_genre)<10: continue
    model = TfidfLogReg()
    model.fit(train_genre['text'].values, train_genre['binary_label'].values)

    test_genre = test_clean[test_clean['style']==genre]
    if len(test_genre):
        res = evaluate(model, test_genre['text'].values, test_genre['binary_label'].values)
        results.append({'exp':'1A_clean_same', 'train':genre, 'test':genre, **res})

    res = evaluate(model, test_clean['text'].values, test_clean['binary_label'].values)
    results.append({'exp':'1B_clean_cross', 'train':genre, 'test':'all', **res})

model_all = TfidfLogReg()
model_all.fit(train_clean['text'].values, train_clean['binary_label'].values)
for genre in genres:
    test_genre = test_clean[test_clean['style']==genre]
    if len(test_genre):
        res = evaluate(model_all, test_genre['text'].values, test_genre['binary_label'].values)
        results.append({'exp':'1C_clean_all', 'train':'all', 'test':genre, **res})


train_all = train_full
test_all = test_full
for genre in genres:
    train_genre = train_all[train_all['style']==genre]
    if len(train_genre)<10: continue
    model = TfidfLogReg()
    model.fit(train_genre['text'].values, train_genre['binary_label'].values)
    test_genre = test_all[test_all['style']==genre]
    if len(test_genre):
        res = evaluate(model, test_genre['text'].values, test_genre['binary_label'].values)
        results.append({'exp':'2A_all_same', 'train':genre, 'test':genre, **res})
    res = evaluate(model, test_all['text'].values, test_all['binary_label'].values)
    results.append({'exp':'2B_all_cross', 'train':genre, 'test':'all', **res})

model_all = TfidfLogReg()
model_all.fit(train_all['text'].values, train_all['binary_label'].values)
for genre in genres:
    test_genre = test_all[test_all['style']==genre]
    if len(test_genre):
        res = evaluate(model_all, test_genre['text'].values, test_genre['binary_label'].values)
        results.append({'exp':'2C_all_train', 'train':'all', 'test':genre, **res})


test_unclean = test_full[~test_full['is_clean']]
for genre in genres:
    train_genre = train_clean[train_clean['style']==genre]
    if len(train_genre)<10: continue
    model = TfidfLogReg()
    model.fit(train_genre['text'].values, train_genre['binary_label'].values)

    test_unclean_genre = test_unclean[test_unclean['style']==genre]
    if len(test_unclean_genre):
        res = evaluate(model, test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values)
        results.append({'exp':'3A_attack_same', 'train':genre, 'test':f'unclean_{genre}', **res})

    res = evaluate(model, test_unclean['text'].values, test_unclean['binary_label'].values)
    results.append({'exp':'3B_attack_cross', 'train':genre, 'test':'all_unclean', **res})

model_all = TfidfLogReg()
model_all.fit(train_clean['text'].values, train_clean['binary_label'].values)
for genre in genres:
    test_unclean_genre = test_unclean[test_unclean['style']==genre]
    if len(test_unclean_genre):
        res = evaluate(model_all, test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values)
        results.append({'exp':'3C_attack_all', 'train':'all_clean', 'test':f'unclean_{genre}', **res})


df_results = pd.DataFrame(results)
print(df_results.to_string())

                exp        train                 test  accuracy     f1_AI  f1_Human
0     1A_clean_same   literature           literature  0.793570  0.762646  0.817365
1    1B_clean_cross   literature                  all  0.793991  0.799283  0.788413
2     1A_clean_same   scientific           scientific  0.854545  0.875000  0.826087
3    1B_clean_cross   scientific                  all  0.751686  0.723549  0.774624
4     1A_clean_same  publicistic          publicistic  0.870000  0.880000  0.858182
5    1B_clean_cross  publicistic                  all  0.795218  0.795343  0.795092
6      1C_clean_all          all           literature  0.825719  0.792757  0.849635
7      1C_clean_all          all           scientific  0.852273  0.879852  0.808260
8      1C_clean_all          all          publicistic  0.875000  0.885145  0.862888
9       2A_all_same   literature           literature  0.806002  0.835005  0.764629
10     2B_all_cross   literature                  all  0.783293  0.827164  0